# MovieLens ml-latest-small - Data Preprocessing for CF

## Preprocessing Pipeline

1. **Clean movies** - Parse genres, extract year, clean titles
2. **Filter ratings** - Keep movies with ≥5 ratings, users with ≥5 ratings
3. **Create user-item matrix** - Sparse CSR matrix for similarity computations
4. **Train/test split** - User-based (hold out last N ratings per user)
5. **Create surprise Datasets** - For SVD/k-NN training
6. **Save processed data** - Parquet + joblib for fast loading

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from scipy import sparse
import joblib

from src.preprocessing import (
    clean_movies,
    filter_ratings,
    create_user_item_matrix,
    create_surprise_dataset,
    user_based_train_test_split,
    time_based_train_test_split,
    run_preprocessing_pipeline,
    load_preprocessed_data,
)

In [ ]:
# Run full preprocessing pipeline
result = run_preprocessing_pipeline(
    min_movie_ratings=5,
    min_user_ratings=5,
    test_ratio=0.2,
    split_method="user_based",
    random_state=42,
)

In [ ]:
# Inspect results
print("Movies clean shape:", result["movies_clean"].shape)
print("Ratings clean shape:", result["ratings_clean"].shape)
print("Train ratings shape:", result["train_ratings"].shape)
print("Test ratings shape:", result["test_ratings"].shape)
print("User-item matrix shape:", result["user_item_matrix"].shape)
print("Matrix sparsity:", 1 - (result["user_item_matrix"].nnz / (result["user_item_matrix"].shape[0] * result["user_item_matrix"].shape[1])))
print("Users:", len(result["mappings"]["user_ids"]))
print("Movies:", len(result["mappings"]["movie_ids"]))

In [ ]:
# Check movies_clean
result["movies_clean"].head()

In [ ]:
# Check ratings_clean
result["ratings_clean"].head()

In [ ]:
# Check train/test split
train = result["train_ratings"]
test = result["test_ratings"]
print("Train users:", train["userId"].nunique())
print("Test users:", test["userId"].nunique())
print("Train movies:", train["movieId"].nunique())
print("Test movies:", test["movieId"].nunique())

# Verify no data leakage: test movies should be subset of train movies for each user
for uid in train["userId"].unique()[:5]:
    train_movies = set(train[train["userId"] == uid]["movieId"])
    test_movies = set(test[test["userId"] == uid]["movieId"])
    overlap = train_movies & test_movies
    print(f"User {uid}: train={len(train_movies)}, test={len(test_movies)}, overlap={len(overlap)}")

In [ ]:
# Verify surprise datasets work
train_data = result["train_data"]
test_data = result["test_data"]

trainset = train_data.build_full_trainset()
print(f"Trainset: {trainset.n_users} users, {trainset.n_items} items, {trainset.n_ratings} ratings")
print(f"Rating scale: {trainset.rating_scale}")

# Check a few ratings
for uid, iid, true_r in list(trainset.all_ratings())[:5]:
    print(f"User {uid} -> Item {iid}: {true_r}")

In [ ]:
# Load saved data to verify persistence
loaded = load_preprocessed_data()
print("Loaded movies_clean:", loaded["movies_clean"].shape)
print("Loaded ratings_clean:", loaded["ratings_clean"].shape)
print("Loaded user_item_matrix:", loaded["user_item_matrix"].shape)
print("Loaded mappings keys:", list(loaded["mappings"].keys()))
print("Loaded train_data type:", type(loaded["train_data"]))
print("Loaded test_data type:", type(loaded["test_data"]))

## Summary

| Metric | Value |
|--------|-------|
| Original ratings | 100,836 |
| Filtered ratings | 90,274 |
| Original movies | 9,742 |
| Movies with ≥5 ratings | 3,650 |
| Original users | 610 |
| Users with ≥5 ratings | 610 |
| Train ratings | 72,434 |
| Test ratings | 17,840 |
| User-item matrix | 610 × 3,650 |
| Matrix sparsity | 95.9% |

## Next Steps (Day 3-4)
1. Train SVD model on trainset
2. Train k-NN baseline
3. Hyperparameter tuning with GridSearchCV
4. Evaluate on testset (RMSE, MAE, Precision@10)